<div dir = 'rtl'>
<H1>
ניקוי נתונים מתוך DataFrame
</H1>
תחילה הצגנו פונקציות שיכולות לסייע לנו לנקות נתונים.
<br/>
לפני מנקים את הנתונים נבדוק את העמודות, נבחר אילו עמודות מעניינות אותי או לא
<br/>
נבדוק סוגי נתונים ולבדוק אם הם עמודות מספריות אבל הם מסוג object ולכן לא נוכל לבצע עליהם חישובים אז נמיר אותם לעמודה מספריות
<br/>
נבדוק שאין לנו שורות שחוזרות על עצמם ונבדוק אם אנחנו מעונינים למחוק אותם
</div>

<div dir = 'rtl'>
נעבוד על dataframe שמרכזת נתונים על סרטים
</div>

In [3]:
# import pandas
import pandas as pd
import os
import os.path

current_directory = os.path.abspath(os.getcwd())
csv_path = os.path.join(current_directory, "movie_metadata.csv")

movies_df = pd.read_csv(csv_path)


In [ ]:
#display how many rows and columns exist in our dataset
movies_df.shape
#look at first 10 rows
movies_df.head(10)
# look at the list of columns and the type of values they are
movies_df.dtypes
# look at dataframe description
movies_df.describe()
#find out number of columns without values
(movies_df.isnull().sum() > 0).sum()

<div dir = 'rtl'>
בואו נסדר קצת את הטבלה כדי שיראה קצת יותר טוב. נשנה קצת כותרות של עמודות ונדאג שיהיה קצת יותר מסודר.<br/>
נדאג שהנתנונים יהיו אחידים כך שיהיה יותר קל לעבד אותם אחר כך.
</div>

In [27]:
# just in case, let us make sure that our movie titles name are all in upper case
movies_df["movie_title"] = movies_df["movie_title"].str.upper()
# remove all extra spaces before and after the movie title
movies_df["movie_title"] = movies_df["movie_title"].str.strip()
#Let us change the name of the column which is far too long:movie_facebook_likes to fb_likes
movies_df.rename(columns={"movie_facebook_likes": "fb_likes"}, inplace=True)

<div dir = 'rtl'>
נבדוק תחילה אם יש לנו בכלל עמודות שחוזרות על עצמם.  נפעיל פונקציה שבודקת האם יש חזרות
אם יש חזרות, נמחק אותם.
תחילה נבדוק האם יש לנו שורות זהות ואחר כך נוודא שאין לנו שורות עם אותה שם של סרט
</div>

In [31]:
# find out if there are duplicate rows
movies_df.duplicated().sum()
#delete them if they exist
movies_df.drop_duplicates(inplace=True)
# find if there are any rows that have the same movie name. 
movies_df.duplicated("movie_title").sum()
#delete them if you find them
movies_df.drop_duplicates("movie_title", inplace=True)

<div dir = 'rtl'>
מצאנו שבחלק מהעמודות חסרות לנו נתונים. בחלק מהעמודות נמלא ערכים שאנחנו חליטים עליהם ובחלקם נמחק את השורות
</div>

In [64]:
# replace all of the rows in which the duration of the movie is missing to the average duration of all 
movies_df["duration"] = movies_df["duration"].fillna(movies_df["duration"].mean())
#remove all the movies in which a year is not given
movies_df["title_year"].dropna()
#remove any columns that all the values are missing
movies_df = movies_df.dropna(axis=1, how="all")
#remove all rows in which the budget or grossincome are missing
movies_df = movies_df.dropna(axis=0, subset=["gross", "budget"], how="any")
#replace all missing countries with USA
movies_df["country"] = movies_df["country"].fillna("USA")
#replace all missing languages with English
movies_df["language"] = movies_df["language"].fillna("English")
#replace all missing ratings with unrated
movies_df["content_rating"] = movies_df["content_rating"].fillna("Unrated")

<div dir = 'rtl'>
נוסיף עוד עמודות:
<br/>
נוסיף עמודה של gross_income שיציג רווח נקי. אנחנו נחסיר את התקציב (budget)מההכנסה (income)
<br/>
נציג את הנתונים המעניינים: ממוצעת הכנסה, ממוצעת תקציב, סטיית תקן של ההוצאה לסרטים וכן הלאה. תחשבו על נתונים שמעניינים אתכם
</div>

In [71]:
movies_df["gross_income"] = movies_df["gross"] - movies_df["budget"]

print("Average gross: ", round(movies_df["gross"].mean(), 2))
print("Average budget: ", round(movies_df["budget"].mean(), 2))

best = movies_df.loc[movies_df["gross_income"].idxmax()]
print("\nMost profitable movie:", best["movie_title"], "->", best["gross_income"])

Average gross:  50136477.25
Average budget:  41443455.45

Most profitable movie: AVATAR -> 523505847.0
